# Qwen2VL Baseline

Notebook baseline cho `ViTextVQA` / `ViOCRVQA` với 2 metric `EM` và `F1`.

Notebook này giữ đúng prompt template theo yêu cầu ở phần config; riêng token ảnh và chat wrapper được render qua `Qwen2-VL` processor để tránh trùng special tokens.

In [ ]:
import os
from huggingface_hub import hf_hub_download

repo_id = "minhquan6203/ViTextVQA"

filenames_to_download = [
    "ViTextVQA_train.json",
    "ViTextVQA_dev.json",
    "ViTextVQA_test_gt.json",
    "ViTextVQA_images.zip"
]
folder = "./CS2230-VQA/data/ViTextVQA"
os.makedirs(folder, exist_ok=True)

for filename in filenames_to_download:
    try:
        downloaded_file_path = hf_hub_download(
            repo_id=repo_id,
            filename=filename,
            repo_type="dataset",
            local_dir=folder
        )
        print(f"Tệp '{filename}' đã được tải xuống tại: {downloaded_file_path}")
    except Exception as e:
        print(f"Lỗi khi tải '{filename}': {e}")

Tệp 'ViTextVQA_train.json' đã được tải xuống tại: CS2230-VQA/data/ViTextVQA/ViTextVQA_train.json
Tệp 'ViTextVQA_dev.json' đã được tải xuống tại: CS2230-VQA/data/ViTextVQA/ViTextVQA_dev.json
Tệp 'ViTextVQA_test_gt.json' đã được tải xuống tại: CS2230-VQA/data/ViTextVQA/ViTextVQA_test_gt.json


ViTextVQA_images.zip:   0%|          | 0.00/5.58G [00:00<?, ?B/s]

Tệp 'ViTextVQA_images.zip' đã được tải xuống tại: CS2230-VQA/data/ViTextVQA/ViTextVQA_images.zip


In [ ]:
!unzip -qq /content/CS2230-VQA/data/ViTextVQA/ViTextVQA_images.zip -d /content/CS2230-VQA/data/ViTextVQA/

In [ ]:
from pathlib import Path

# =========================
# Central Config
# =========================
DATA_ROOT = Path("/content/CS2230-VQA/data")
DATASET_NAME = "ViTextVQA"  # "ViTextVQA" | "ViOCRVQA"
DATASET_DIR = DATA_ROOT / DATASET_NAME
IMAGE_DIR = DATASET_DIR / "st_images"      # override nếu folder ảnh khác tên
TEST_JSON = DATASET_DIR / "ViTextVQA_test_gt.json" # override nếu file test khác tên

MODEL_VARIANT = "2B"  # "2B", "7B" hoặc nhập trực tiếp HF model id
MODEL_NAME = {
    "2B": "Qwen/Qwen2-VL-2B-Instruct",
    "7B": "Qwen/Qwen2-VL-7B-Instruct",
}.get(MODEL_VARIANT, MODEL_VARIANT)

DEVICE = "auto"  # "auto" | "cuda" | "cpu"

OUTPUT_DIR = Path("baseline") / "outputs" / f"qwen2vl_{DATASET_NAME.lower()}_{MODEL_VARIANT.lower().replace('/', '_')}"
PREDICTIONS_PATH = OUTPUT_DIR / "predictions.json"
STATS_PATH = OUTPUT_DIR / "stats.json"

BATCH_SIZE = 4
MAX_SAMPLES = None        # None = chạy toàn bộ test set
SKIP_MISSING_IMAGES = False
SEED = 42

TEMPERATURE = 0.15        # khuyến nghị: 0.1 - 0.2
TOP_P = 0.85              # khuyến nghị: 0.8 - 0.9
MAX_NEW_TOKENS = 24       # khuyến nghị: 16 - 32
DO_SAMPLE = True

MIN_PIXELS = 256 * 28 * 28
MAX_PIXELS = 512 * 28 * 28
IMAGE_EXTS = [".jpg", ".jpeg", ".png", ".webp", ".bmp"]

PROMPT_TEMPLATE = """<image>
User: Trả lời câu hỏi VQA tiếng Việt dựa trên nội dung trong ảnh.
Hãy trả lời ngắn gọn chỉ bằng một từ, cụm từ hoặc số, không giải thích thêm.
Câu hỏi: {question}
Assistant:"""

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CONFIG = {
    "dataset_name": DATASET_NAME,
    "dataset_dir": str(DATASET_DIR),
    "image_dir": str(IMAGE_DIR),
    "test_json": str(TEST_JSON),
    "model_variant": MODEL_VARIANT,
    "model_name": MODEL_NAME,
    "device": DEVICE,
    "batch_size": BATCH_SIZE,
    "max_samples": MAX_SAMPLES,
    "skip_missing_images": SKIP_MISSING_IMAGES,
    "seed": SEED,
    "temperature": TEMPERATURE,
    "top_p": TOP_P,
    "max_new_tokens": MAX_NEW_TOKENS,
    "do_sample": DO_SAMPLE,
    "min_pixels": MIN_PIXELS,
    "max_pixels": MAX_PIXELS,
    "predictions_path": str(PREDICTIONS_PATH),
    "stats_path": str(STATS_PATH),
}
CONFIG

{'dataset_name': 'ViTextVQA',
 'dataset_dir': '/content/CS2230-VQA/data/ViTextVQA',
 'image_dir': '/content/CS2230-VQA/data/ViTextVQA/st_images',
 'test_json': '/content/CS2230-VQA/data/ViTextVQA/ViTextVQA_test_gt.json',
 'model_variant': '2B',
 'model_name': 'Qwen/Qwen2-VL-2B-Instruct',
 'device': 'auto',
 'batch_size': 4,
 'max_samples': None,
 'skip_missing_images': False,
 'seed': 42,
 'temperature': 0.15,
 'top_p': 0.85,
 'max_new_tokens': 24,
 'do_sample': True,
 'min_pixels': 200704,
 'max_pixels': 401408,
 'predictions_path': 'baseline/outputs/qwen2vl_vitextvqa_2b/predictions.json',
 'stats_path': 'baseline/outputs/qwen2vl_vitextvqa_2b/stats.json'}

In [ ]:
# @title
import base64
import copy
import logging
import math
import os
import sys
import time
import warnings
from functools import lru_cache
from io import BytesIO
from typing import Optional, Union, Tuple, List, Any, Dict
from concurrent.futures import ThreadPoolExecutor

import requests
import torch
import torchvision
from packaging import version
from PIL import Image
import numpy as np
from torchvision import io, transforms
from torchvision.transforms import InterpolationMode


MAX_RATIO = 200
SPATIAL_MERGE_SIZE = 2
IMAGE_MIN_TOKEN_NUM = 4
IMAGE_MAX_TOKEN_NUM = 16384
VIDEO_MIN_TOKEN_NUM = 128
VIDEO_MAX_TOKEN_NUM = 768

FPS = 2.0
FRAME_FACTOR = 2
FPS_MIN_FRAMES = 4
FPS_MAX_FRAMES = 768
MAX_NUM_WORKERS_FETCH_VIDEO = 8

MODEL_SEQ_LEN = int(float(os.environ.get('MODEL_SEQ_LEN', 128000)))
logger = logging.getLogger(__name__)


def round_by_factor(number: int, factor: int) -> int:
    """Returns the closest integer to 'number' that is divisible by 'factor'."""
    return round(number / factor) * factor


def ceil_by_factor(number: int, factor: int) -> int:
    """Returns the smallest integer greater than or equal to 'number' that is divisible by 'factor'."""
    return math.ceil(number / factor) * factor


def floor_by_factor(number: int, factor: int) -> int:
    """Returns the largest integer less than or equal to 'number' that is divisible by 'factor'."""
    return math.floor(number / factor) * factor


def smart_resize(height: int, width: int, factor: int, min_pixels: Optional[int] = None, max_pixels: Optional[int] = None) -> Tuple[int, int]:
    """
    Rescales the image so that the following conditions are met:

    1. Both dimensions (height and width) are divisible by 'factor'.
    2. The total number of pixels is within the range ['min_pixels', 'max_pixels'].
    3. The aspect ratio of the image is maintained as closely as possible.
    """
    max_pixels = max_pixels if max_pixels is not None else (IMAGE_MAX_TOKEN_NUM * factor ** 2)
    min_pixels = min_pixels if min_pixels is not None else (IMAGE_MIN_TOKEN_NUM * factor ** 2)
    assert max_pixels >= min_pixels, "The max_pixels of image must be greater than or equal to min_pixels."
    if max(height, width) / min(height, width) > MAX_RATIO:
        raise ValueError(
            f"absolute aspect ratio must be smaller than {MAX_RATIO}, got {max(height, width) / min(height, width)}"
        )
    h_bar = max(factor, round_by_factor(height, factor))
    w_bar = max(factor, round_by_factor(width, factor))
    if h_bar * w_bar > max_pixels:
        beta = math.sqrt((height * width) / max_pixels)
        h_bar = floor_by_factor(height / beta, factor)
        w_bar = floor_by_factor(width / beta, factor)
    elif h_bar * w_bar < min_pixels:
        beta = math.sqrt(min_pixels / (height * width))
        h_bar = ceil_by_factor(height * beta, factor)
        w_bar = ceil_by_factor(width * beta, factor)
    return h_bar, w_bar


def to_rgb(pil_image: Image.Image) -> Image.Image:
      if pil_image.mode == 'RGBA':
          white_background = Image.new("RGB", pil_image.size, (255, 255, 255))
          white_background.paste(pil_image, mask=pil_image.split()[3])  # Use alpha channel as mask
          return white_background
      else:
          return pil_image.convert("RGB")


def fetch_image(ele: Dict[str, Union[str, Image.Image]], image_patch_size: int = 14) -> Image.Image:
    if "image" in ele:
        image = ele["image"]
    else:
        image = ele["image_url"]

    image_obj = None
    patch_factor = int(image_patch_size * SPATIAL_MERGE_SIZE)
    if isinstance(image, Image.Image):
        image_obj = image
    elif image.startswith("http://") or image.startswith("https://"):
        with requests.get(image, stream=True) as response:
            response.raise_for_status()
            with BytesIO(response.content) as bio:
                image_obj = copy.deepcopy(Image.open(bio))
    elif image.startswith("file://"):
        image_obj = Image.open(image[7:])
    elif image.startswith("data:image"):
        if "base64," in image:
            _, base64_data = image.split("base64,", 1)
            data = base64.b64decode(base64_data)
            with BytesIO(data) as bio:
                image_obj = copy.deepcopy(Image.open(bio))
    else:
        image_obj = Image.open(image)
    if image_obj is None:
        raise ValueError(f"Unrecognized image input, support local path, http url, base64 and PIL.Image, got {image}")
    image = to_rgb(image_obj)

    ## resize
    if "resized_height" in ele and "resized_width" in ele:
        resized_height, resized_width = smart_resize(
            ele["resized_height"],
            ele["resized_width"],
            factor=patch_factor,
        )
    else:
        width, height = image.size
        min_pixels = ele.get("min_pixels", IMAGE_MIN_TOKEN_NUM * patch_factor ** 2)
        max_pixels = ele.get("max_pixels", IMAGE_MAX_TOKEN_NUM * patch_factor ** 2)
        resized_height, resized_width = smart_resize(
            height,
            width,
            factor=patch_factor,
            min_pixels=min_pixels,
            max_pixels=max_pixels,
        )
    image = image.resize((resized_width, resized_height))
    return image


def smart_nframes(
    ele: Dict[str, Any],
    total_frames: int,
    video_fps: Union[int, float],
) -> int:
    """calculate the number of frames for video used for model inputs.

    Args:
        ele (dict): a dict contains the configuration of video.
            support either `fps` or `nframes`:
                - nframes: the number of frames to extract for model inputs.
                - fps: the fps to extract frames for model inputs.
                    - min_frames: the minimum number of frames of the video, only used when fps is provided.
                    - max_frames: the maximum number of frames of the video, only used when fps is provided.
        total_frames (int): the original total number of frames of the video.
        video_fps (int | float): the original fps of the video.

    Raises:
        ValueError: nframes should in interval [FRAME_FACTOR, total_frames].

    Returns:
        int: the number of frames for video used for model inputs.
    """
    assert not ("fps" in ele and "nframes" in ele), "Only accept either `fps` or `nframes`"
    if "nframes" in ele:
        nframes = round_by_factor(ele["nframes"], FRAME_FACTOR)
    else:
        fps = ele.get("fps", FPS)
        min_frames = ceil_by_factor(ele.get("min_frames", FPS_MIN_FRAMES), FRAME_FACTOR)
        max_frames = floor_by_factor(ele.get("max_frames", min(FPS_MAX_FRAMES, total_frames)), FRAME_FACTOR)
        nframes = total_frames / video_fps * fps
        if nframes > total_frames:
            logger.warning(f"smart_nframes: nframes[{nframes}] > total_frames[{total_frames}]")
        nframes = min(min(max(nframes, min_frames), max_frames), total_frames)
        nframes = floor_by_factor(nframes, FRAME_FACTOR)
    if not (FRAME_FACTOR <= nframes and nframes <= total_frames):
        raise ValueError(f"nframes should in interval [{FRAME_FACTOR}, {total_frames}], but got {nframes}.")
    return nframes


def _read_video_torchvision(
    ele: Dict[str, Any],
) -> Tuple[torch.Tensor, float]:
    """read video using torchvision.io.read_video

    Args:
        ele (dict): a dict contains the configuration of video.
        support keys:
            - video: the path of video. support "file://", "http://", "https://" and local path.
            - video_start: the start time of video.
            - video_end: the end time of video.
    Returns:
        torch.Tensor: the video tensor with shape (T, C, H, W).
    """
    video_path = ele["video"]
    if version.parse(torchvision.__version__) < version.parse("0.19.0"):
        if "http://" in video_path or "https://" in video_path:
            warnings.warn("torchvision < 0.19.0 does not support http/https video path, please upgrade to 0.19.0.")
        if "file://" in video_path:
            video_path = video_path[7:]
    st = time.time()
    video, audio, info = io.read_video(
        video_path,
        start_pts=ele.get("video_start", 0.0),
        end_pts=ele.get("video_end", None),
        pts_unit="sec",
        output_format="TCHW",
    )
    total_frames, video_fps = video.size(0), info["video_fps"]
    logger.info(f"torchvision:  {video_path=}, {total_frames=}, {video_fps=}, time={time.time() - st:.3f}s")
    nframes = smart_nframes(ele, total_frames=total_frames, video_fps=video_fps)
    idx = torch.linspace(0, total_frames - 1, nframes).round().long()
    sample_fps = nframes / max(total_frames, 1e-6) * video_fps
    video = video[idx]

    video_metadata = dict(
        fps=video_fps,
        frames_indices=idx,
        total_num_frames=total_frames,
        video_backend="torchvision",
    )
    return video, video_metadata, sample_fps


def is_decord_available() -> bool:
    import importlib.util

    return importlib.util.find_spec("decord") is not None


def calculate_video_frame_range(
    ele: Dict[str, Any],
    total_frames: int,
    video_fps: float,
) -> Tuple[int, int, int]:
    """
    Calculate the start and end frame indices based on the given time range.

    Args:
        ele (dict): A dictionary containing optional 'video_start' and 'video_end' keys (in seconds).
        total_frames (int): Total number of frames in the video.
        video_fps (float): Frames per second of the video.

    Returns:
        tuple: A tuple containing (start_frame, end_frame, frame_count).

    Raises:
        ValueError: If input parameters are invalid or the time range is inconsistent.
    """
    # Validate essential parameters
    if video_fps <= 0:
        raise ValueError("video_fps must be a positive number")
    if total_frames <= 0:
        raise ValueError("total_frames must be a positive integer")

    # Get start and end time in seconds
    video_start = ele.get("video_start", None)
    video_end = ele.get("video_end", None)
    if video_start is None and video_end is None:
        return 0, total_frames - 1, total_frames

    max_duration = total_frames / video_fps
    # Process start frame
    if video_start is not None:
        video_start_clamped = max(0.0, min(video_start, max_duration))
        start_frame = math.ceil(video_start_clamped * video_fps)
    else:
        start_frame = 0
    # Process end frame
    if video_end is not None:
        video_end_clamped = max(0.0, min(video_end, max_duration))
        end_frame = math.floor(video_end_clamped * video_fps)
        end_frame = min(end_frame, total_frames - 1)
    else:
        end_frame = total_frames - 1

    # Validate frame order
    if start_frame >= end_frame:
        raise ValueError(
            f"Invalid time range: Start frame {start_frame} (at {video_start_clamped if video_start is not None else 0}s) "
            f"exceeds end frame {end_frame} (at {video_end_clamped if video_end is not None else max_duration}s). "
            f"Video duration: {max_duration:.2f}s ({total_frames} frames @ {video_fps}fps)"
        )

    logger.info(f"calculate video frame range: {start_frame=}, {end_frame=}, {total_frames=} from {video_start=}, {video_end=}, {video_fps=:.3f}")
    return start_frame, end_frame, end_frame - start_frame + 1


def _read_video_decord(
    ele: Dict[str, Any],
) -> Tuple[torch.Tensor, float]:
    """read video using decord.VideoReader

    Args:
        ele (dict): a dict contains the configuration of video.
        support keys:
            - video: the path of video. support "file://", "http://", "https://" and local path.
            - video_start: the start time of video.
            - video_end: the end time of video.
    Returns:
        torch.Tensor: the video tensor with shape (T, C, H, W).
    """
    import decord
    video_path = ele["video"]
    st = time.time()
    vr = decord.VideoReader(video_path)
    total_frames, video_fps = len(vr), vr.get_avg_fps()
    start_frame, end_frame, total_frames = calculate_video_frame_range(
        ele,
        total_frames,
        video_fps,
    )
    nframes = smart_nframes(ele, total_frames=total_frames, video_fps=video_fps)
    idx = torch.linspace(start_frame, end_frame, nframes).round().long().tolist()
    video = vr.get_batch(idx).asnumpy()
    video = torch.tensor(video).permute(0, 3, 1, 2)  # Convert to TCHW format
    logger.info(f"decord:  {video_path=}, {total_frames=}, {video_fps=}, time={time.time() - st:.3f}s")
    sample_fps = nframes / max(total_frames, 1e-6) * video_fps

    video_metadata = dict(
        fps=video_fps,
        frames_indices=idx,
        total_num_frames=total_frames,
        video_backend="decord",
    )
    return video, video_metadata, sample_fps


def is_torchcodec_available() -> bool:
    import importlib.util

    return importlib.util.find_spec("torchcodec") is not None


def _read_video_torchcodec(
    ele: Dict[str, Any],
) -> Tuple[torch.Tensor, float]:
    """read video using torchcodec.decoders.VideoDecoder

    Args:
        ele (dict): a dict contains the configuration of video.
        support keys:
            - video: the path of video. support "file://", "http://", "https://" and local path.
            - video_start: the start time of video.
            - video_end: the end time of video.
    Returns:
        torch.Tensor: the video tensor with shape (T, C, H, W).
    """
    from torchcodec.decoders import VideoDecoder
    TORCHCODEC_NUM_THREADS = int(os.environ.get('TORCHCODEC_NUM_THREADS', 8))
    logger.info(f"set TORCHCODEC_NUM_THREADS: {TORCHCODEC_NUM_THREADS}")
    video_path = ele["video"]
    st = time.time()
    decoder = VideoDecoder(video_path, num_ffmpeg_threads=TORCHCODEC_NUM_THREADS)
    video_fps = decoder.metadata.average_fps
    total_frames = decoder.metadata.num_frames
    start_frame, end_frame, total_frames = calculate_video_frame_range(
        ele,
        total_frames,
        video_fps,
    )
    nframes = smart_nframes(ele, total_frames=total_frames, video_fps=video_fps)
    idx = torch.linspace(start_frame, end_frame, nframes).round().long().tolist()
    sample_fps = nframes / max(total_frames, 1e-6) * video_fps
    video = decoder.get_frames_at(indices=idx).data
    logger.info(f"torchcodec:  {video_path=}, {total_frames=}, {video_fps=}, time={time.time() - st:.3f}s")

    video_metadata = dict(
        fps=video_fps,
        frames_indices=idx,
        total_num_frames=total_frames,
        video_backend="torchcodec",
    )
    return video, video_metadata, sample_fps


VIDEO_READER_BACKENDS = {
    "decord": _read_video_decord,
    "torchvision": _read_video_torchvision,
    "torchcodec": _read_video_torchcodec,
}

FORCE_QWENVL_VIDEO_READER = os.getenv("FORCE_QWENVL_VIDEO_READER", None)


@lru_cache(maxsize=1)
def get_video_reader_backend() -> str:
    if FORCE_QWENVL_VIDEO_READER is not None:
        video_reader_backend = FORCE_QWENVL_VIDEO_READER
    elif is_torchcodec_available():
        video_reader_backend = "torchcodec"
    elif is_decord_available():
        video_reader_backend = "decord"
    else:
        video_reader_backend = "torchvision"
    print(f"qwen-vl-utils using {video_reader_backend} to read video.", file=sys.stderr)
    return video_reader_backend


def fetch_video(ele: Dict[str, Any], image_patch_size: int = 14, return_video_sample_fps: bool = False,
                return_video_metadata: bool = False) -> Union[torch.Tensor, List[Image.Image]]:
    image_factor = image_patch_size * SPATIAL_MERGE_SIZE
    VIDEO_FRAME_MIN_PIXELS = VIDEO_MIN_TOKEN_NUM * image_factor * image_factor
    VIDEO_FRAME_MAX_PIXELS = VIDEO_MAX_TOKEN_NUM * image_factor * image_factor
    if isinstance(ele["video"], str):
        video_reader_backend = get_video_reader_backend()
        try:
            video, video_metadata, sample_fps = VIDEO_READER_BACKENDS[video_reader_backend](ele)
        except Exception as e:
            logger.warning(f"video_reader_backend {video_reader_backend} error, use torchvision as default, msg: {e}")
            video, video_metadata, sample_fps = VIDEO_READER_BACKENDS["torchvision"](ele)
    else:
        # The input is a list of frames
        assert isinstance(ele["video"], (list, tuple))
        process_info = ele.copy()
        process_info.pop("type", None)
        process_info.pop("video", None)
        # use ThreadPoolExecutor to parallel process frames
        max_workers = min(MAX_NUM_WORKERS_FETCH_VIDEO, len(ele["video"]))
        with ThreadPoolExecutor(max_workers=max_workers) as executor:
            futures = [
                executor.submit(fetch_image, {"image": video_element, **process_info}, image_patch_size)
                for video_element in ele["video"]
            ]
            image_list = [future.result() for future in futures]

        nframes = ceil_by_factor(len(image_list), FRAME_FACTOR)
        if len(image_list) < nframes:
            image_list.extend([image_list[-1]] * (nframes - len(image_list)))

        sample_fps = ele.get("sample_fps", 2.0)
        video = torch.stack([
            torch.from_numpy(np.array(image).transpose(2, 0, 1))
            for image in image_list
        ])

        # fake video metadata
        raw_fps = process_info.pop("raw_fps", sample_fps)
        video_metadata = dict(
            fps=raw_fps,
            frames_indices=[i for i in range(len(video))],
            total_num_frames=(nframes / sample_fps) * raw_fps,
        )

    nframes, _, height, width = video.shape
    min_pixels = ele.get("min_pixels", VIDEO_FRAME_MIN_PIXELS)
    total_pixels = ele.get("total_pixels", MODEL_SEQ_LEN * image_factor * image_factor * 0.9)
    max_pixels = max(min(VIDEO_FRAME_MAX_PIXELS, total_pixels / nframes * FRAME_FACTOR), int(min_pixels * 1.05))
    max_pixels_supposed = ele.get("max_pixels", max_pixels)
    if max_pixels_supposed > max_pixels:
        logger.warning(f"The given max_pixels[{max_pixels_supposed}] exceeds limit[{max_pixels}].")
    max_pixels = min(max_pixels_supposed, max_pixels)
    if "resized_height" in ele and "resized_width" in ele:
        resized_height, resized_width = smart_resize(
            ele["resized_height"],
            ele["resized_width"],
            factor=image_factor,
        )
    else:
        resized_height, resized_width = smart_resize(
            height,
            width,
            factor=image_factor,
            min_pixels=min_pixels,
            max_pixels=max_pixels,
        )
    video = transforms.functional.resize(
        video,
        [resized_height, resized_width],
        interpolation=InterpolationMode.BICUBIC,
        antialias=True,
    ).float()

    final_video = (video, video_metadata) if return_video_metadata else video
    if return_video_sample_fps:
        return final_video, sample_fps
    return final_video


def extract_vision_info(conversations: Union[List[Dict[str, Any]], List[List[Dict[str, Any]]]]) -> List[Dict[str, Any]]:
    vision_infos = []
    if isinstance(conversations[0], dict):
        conversations = [conversations]
    for conversation in conversations:
        for message in conversation:
            if isinstance(message["content"], list):
                for ele in message["content"]:
                    if (
                        "image" in ele
                        or "image_url" in ele
                        or "video" in ele
                        or ele.get("type", "text") in ("image", "image_url", "video")
                    ):
                        vision_infos.append(ele)
    return vision_infos


def process_vision_info(
    conversations: Union[List[Dict[str, Any]], List[List[Dict[str, Any]]]],
    return_video_kwargs: bool = False,
    return_video_metadata: bool = False,
    image_patch_size: int = 14,
) -> Tuple[Optional[List[Image.Image]], Optional[List[Union[torch.Tensor, List[Image.Image]]]], Optional[Dict[str, Any]]]:

    vision_infos = extract_vision_info(conversations)
    ## Read images or videos
    image_inputs = []
    video_inputs = []
    video_sample_fps_list = []
    for vision_info in vision_infos:
        if "image" in vision_info or "image_url" in vision_info:
            image_inputs.append(fetch_image(vision_info, image_patch_size=image_patch_size))
        elif "video" in vision_info:
            video_input, video_sample_fps = fetch_video(vision_info, return_video_sample_fps=True,
                        image_patch_size=image_patch_size, return_video_metadata=return_video_metadata)
            video_sample_fps_list.append(video_sample_fps)
            video_inputs.append(video_input)
        else:
            raise ValueError("image, image_url or video should in content.")
    if len(image_inputs) == 0:
        image_inputs = None
    if len(video_inputs) == 0:
        video_inputs = None

    video_kwargs = {'do_sample_frames': False}
    if not return_video_metadata: # BC for qwen2.5vl
        video_kwargs.update({'fps': video_sample_fps_list})

    if return_video_kwargs:
        return image_inputs, video_inputs, video_kwargs
    return image_inputs, video_inputs

In [ ]:
import json
import random
import re
import unicodedata
from collections import Counter
from functools import lru_cache

import numpy as np
import torch
from PIL import Image
from tqdm.auto import tqdm
from transformers import AutoProcessor, Qwen2VLForConditionalGeneration


def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(SEED)
print(f"Seed set to {SEED}")

Seed set to 42


In [ ]:
# @title
ZERO_WIDTH_RE = re.compile(r"[\u200b\u200c\u200d\ufeff]")
EDGE_PUNCT_RE = re.compile(r"^[\s\.,!?;:'\"“”‘’`~_]+|[\s\.,!?;:'\"“”‘’`~_]+$")
DASH_TRANSLATION = str.maketrans({"–": "-", "—": "-", "−": "-", "‐": "-"})
SIMPLE_NUMBER_WORDS = {
    "không": "0",
    "một": "1",
    "hai": "2",
    "ba": "3",
    "bốn": "4",
    "tư": "4",
    "năm": "5",
    "lăm": "5",
    "sáu": "6",
    "bảy": "7",
    "bẩy": "7",
    "tám": "8",
    "chín": "9",
    "mười": "10",
    "zero": "0",
    "one": "1",
    "two": "2",
    "three": "3",
    "four": "4",
    "five": "5",
    "six": "6",
    "seven": "7",
    "eight": "8",
    "nine": "9",
    "ten": "10",
}


def _canonicalize_number(text: str) -> str:
    compact = text.replace(" ", "")
    if re.fullmatch(r"\d{1,3}([.,]\d{3})+", compact):
        return re.sub(r"[.,]", "", compact)
    if re.fullmatch(r"\d+,\d+", compact) and not re.fullmatch(r"\d{1,3}(,\d{3})+", compact):
        return compact.replace(",", ".")
    return text


def normalize_answer(text: str) -> str:
    """Chuẩn hóa text trước khi so sánh."""
    if text is None:
        return ""

    text = str(text)
    text = unicodedata.normalize("NFC", text)
    text = text.translate(DASH_TRANSLATION)
    text = ZERO_WIDTH_RE.sub("", text)
    text = text.lower().strip()
    text = re.sub(r"\s+", " ", text)
    text = EDGE_PUNCT_RE.sub("", text)
    text = re.sub(r"[^\w\s\u00C0-\u024F\u1E00-\u1EFF\-\./,]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    if text in SIMPLE_NUMBER_WORDS:
        text = SIMPLE_NUMBER_WORDS[text]

    text = _canonicalize_number(text)
    text = re.sub(r"^[\.,!?;:]+|[\.,!?;:]+$", "", text).strip()
    text = re.sub(r"\s+", " ", text)
    return text


def exact_match_score(prediction: str, ground_truths: list[str]) -> float:
    pred = normalize_answer(prediction)
    return float(any(normalize_answer(gt) == pred for gt in ground_truths))


def _token_f1_single(prediction: str, ground_truth: str) -> float:
    pred_tokens = normalize_answer(prediction).split()
    gt_tokens = normalize_answer(ground_truth).split()

    if not pred_tokens and not gt_tokens:
        return 1.0
    if not pred_tokens or not gt_tokens:
        return 0.0

    common = Counter(pred_tokens) & Counter(gt_tokens)
    overlap = sum(common.values())
    if overlap == 0:
        return 0.0

    precision = overlap / len(pred_tokens)
    recall = overlap / len(gt_tokens)
    return 2 * precision * recall / (precision + recall)


def f1_score(prediction: str, ground_truths: list[str]) -> float:
    return max((_token_f1_single(prediction, gt) for gt in ground_truths), default=0.0)


def compute_metrics(predictions: list[str], ground_truths: list[list[str]]) -> dict:
    assert len(predictions) == len(ground_truths)
    if not predictions:
        return {"exact_match": 0.0, "f1": 0.0, "num_samples": 0}

    em = sum(exact_match_score(pred, gts) for pred, gts in zip(predictions, ground_truths))
    f1 = sum(f1_score(pred, gts) for pred, gts in zip(predictions, ground_truths))
    n = len(predictions)
    return {
        "exact_match": em / n,
        "f1": f1 / n,
        "num_samples": n,
    }


def load_annotations(json_path: Path):
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    if isinstance(data, list):
        return data
    if isinstance(data, dict):
        for key in ["annotations", "data", "items", "samples"]:
            if key in data and isinstance(data[key], list):
                return data[key]
    raise ValueError(f"Unsupported annotation format: {json_path}")


def extract_question(ann: dict) -> str:
    for key in ["question", "query", "text"]:
        if ann.get(key):
            return str(ann[key])
    raise KeyError(f"Cannot find question field in sample: {ann}")


def extract_ground_truths(ann: dict) -> list[str]:
    if isinstance(ann.get("answers"), list):
        return [str(x) for x in ann["answers"]]
    if isinstance(ann.get("all_answers"), list):
        return [str(x) for x in ann["all_answers"]]
    if ann.get("answer") is not None:
        return [str(ann["answer"])]
    if ann.get("label") is not None:
        if isinstance(ann["label"], list):
            return [str(x) for x in ann["label"]]
        return [str(ann["label"])]
    return [""]


def extract_image_identifier(ann: dict) -> str:
    for key in ["image_id", "image", "image_name", "file_name", "filename"]:
        value = ann.get(key)
        if value is not None:
            return str(value)
    raise KeyError(f"Cannot find image identifier field in sample: {ann}")


@lru_cache(maxsize=100000)
def resolve_image_path(image_identifier: str, image_dir: str, image_exts: tuple[str, ...]):
    image_dir = Path(image_dir)
    candidate = Path(image_identifier)

    if candidate.suffix:
        direct = image_dir / candidate.name
        if direct.exists():
            return direct
        if candidate.exists():
            return candidate

    stem = candidate.stem if candidate.suffix else candidate.name
    for ext in image_exts:
        path = image_dir / f"{stem}{ext}"
        if path.exists():
            return path

    matches = sorted(image_dir.glob(f"{stem}.*"))
    if matches:
        return matches[0]

    raise FileNotFoundError(f"Cannot find image for id '{image_identifier}' in {image_dir}")


def build_prompt_for_log(question: str) -> str:
    return PROMPT_TEMPLATE.format(question=question)


def build_user_text(question: str) -> str:
    # Qwen2-VL render phần image token và role wrappers qua chat template,
    # nên chỉ truyền phần text instruction vào content để tránh duplicate special tokens.
    raw_prompt = build_prompt_for_log(question).strip()
    raw_prompt = raw_prompt.replace("<image>\n", "", 1)
    raw_prompt = re.sub(r"^User:\s*", "", raw_prompt)
    raw_prompt = re.sub(r"\nAssistant:\s*$", "", raw_prompt)
    return raw_prompt.strip()


print(normalize_answer("  Một—nghìn  "))
print(normalize_answer("1.000"))

một-nghìn
1000


In [ ]:
# @title
if DEVICE == "auto":
    resolved_device = "cuda" if torch.cuda.is_available() else "cpu"
else:
    resolved_device = DEVICE

if resolved_device == "cuda" and not torch.cuda.is_available():
    raise RuntimeError("DEVICE='cuda' nhưng máy hiện tại không có CUDA.")

use_device_map = DEVICE == "auto" and resolved_device == "cuda"
torch_dtype = torch.float32 if resolved_device == "cpu" else (torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16)

print(f"Loading processor: {MODEL_NAME}")
processor = AutoProcessor.from_pretrained(
    MODEL_NAME,
    min_pixels=MIN_PIXELS,
    max_pixels=MAX_PIXELS,
)

print(f"Loading model: {MODEL_NAME}")
model = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch_dtype,
    device_map="auto" if use_device_map else None,
)
if not use_device_map:
    model = model.to(resolved_device)

model.eval()
target_device = next(model.parameters()).device
pad_token_id = processor.tokenizer.pad_token_id
if pad_token_id is None:
    pad_token_id = processor.tokenizer.eos_token_id

print({
    "resolved_device": str(target_device),
    "torch_dtype": str(torch_dtype),
    "use_device_map": use_device_map,
    "pad_token_id": pad_token_id,
})

Loading processor: Qwen/Qwen2-VL-2B-Instruct


preprocessor_config.json:   0%|          | 0.00/347 [00:00<?, ?B/s]

chat_template.json: 0.00B [00:00, ?B/s]

The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/272 [00:00<?, ?B/s]

{'resolved_device': 'cuda:0', 'torch_dtype': 'torch.bfloat16', 'use_device_map': True, 'pad_token_id': 151643}


## Inference Helpers

Batching bên dưới bám theo logic adapter `Qwen2-VL` trong repo: encode từng sample rồi pad `input_ids`/`attention_mask`, còn `pixel_values` và `image_grid_thw` được concat riêng để tránh mismatch khi số image patches khác nhau giữa các ảnh.

In [ ]:
# @title
def pad_1d(sequences: list[torch.Tensor], pad_value: int, max_len: int) -> torch.Tensor:
    padded = []
    for seq in sequences:
        pad_len = max_len - seq.size(0)
        padded.append(torch.cat([seq, torch.full((pad_len,), pad_value, dtype=seq.dtype)]))
    return torch.stack(padded)


def build_messages(question: str, image):
    return [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": build_user_text(question)},
            ],
        }
    ]


def collate_qwen2vl(items: list[dict]) -> dict:
    all_input_ids, all_masks = [], []
    all_pixel_values, all_grid_thw = [], []

    for item in items:
        messages = build_messages(item["question"], item["image"])
        prompt_text = processor.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )
        image_inputs, _ = process_vision_info(messages)
        enc = processor(
            text=[prompt_text],
            images=image_inputs,
            padding=False,
            return_tensors="pt",
        )

        all_input_ids.append(enc.input_ids[0])
        all_masks.append(enc.attention_mask[0])
        all_pixel_values.append(enc.pixel_values)
        all_grid_thw.append(enc.image_grid_thw)

    max_len = max(x.size(0) for x in all_input_ids)
    return {
        "input_ids": pad_1d(all_input_ids, pad_token_id, max_len),
        "attention_mask": pad_1d(all_masks, 0, max_len),
        "pixel_values": torch.cat(all_pixel_values, dim=0),
        "image_grid_thw": torch.cat(all_grid_thw, dim=0),
    }


def move_to_device(inputs: dict, device: torch.device) -> dict:
    return {
        k: (v.to(device) if isinstance(v, torch.Tensor) else v)
        for k, v in inputs.items()
    }


@torch.no_grad()
def generate_batch(batch_items: list[dict]) -> list[str]:
    inputs = collate_qwen2vl(batch_items)
    inputs = move_to_device(inputs, target_device)
    input_len = inputs["input_ids"].shape[1]

    generated = model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        temperature=TEMPERATURE,
        top_p=TOP_P,
        do_sample=DO_SAMPLE,
        num_beams=1,
        pad_token_id=pad_token_id,
    )

    return [
        processor.tokenizer.decode(seq[input_len:], skip_special_tokens=True).strip()
        for seq in generated
    ]

In [ ]:
annotations = load_annotations(TEST_JSON)

In [ ]:
annotations = load_annotations(TEST_JSON)
if MAX_SAMPLES is not None:
    annotations = annotations[:MAX_SAMPLES]

print(f"Loaded {len(annotations):,} test samples from {TEST_JSON}")

results = []
skipped = []
image_exts_tuple = tuple(IMAGE_EXTS)

for start in tqdm(range(0, 10, BATCH_SIZE), desc="Running baseline"):
    batch_anns = annotations[start:start + BATCH_SIZE]
    batch_items = []

    for ann in batch_anns:
        question = extract_question(ann)
        ground_truths = extract_ground_truths(ann)
        image_identifier = extract_image_identifier(ann)
        question_id = ann.get("id", ann.get("question_id", len(results)))

        try:
            image_path = resolve_image_path(image_identifier, str(IMAGE_DIR), image_exts_tuple)
        except FileNotFoundError as exc:
            if SKIP_MISSING_IMAGES:
                skipped.append({
                    "question_id": question_id,
                    "image_identifier": image_identifier,
                    "error": str(exc),
                })
                continue
            raise

        image = Image.open(image_path).convert("RGB")
        batch_items.append({
            "question_id": question_id,
            "image_id": ann.get("image_id", image_identifier),
            "image_path": str(image_path),
            "question": question,
            "ground_truths": ground_truths,
            "image": image,
            "prompt": build_prompt_for_log(question),
        })

    if not batch_items:
        continue

    predictions = generate_batch(batch_items)

    for item, prediction in zip(batch_items, predictions):
        em = exact_match_score(prediction, item["ground_truths"])
        f1 = f1_score(prediction, item["ground_truths"])
        record = {
            "question_id": item["question_id"],
            "image_id": item["image_id"],
            "image_path": item["image_path"],
            "question": item["question"],
            "prompt": item["prompt"],
            "prediction": prediction,
            "prediction_normalized": normalize_answer(prediction),
            "ground_truths": item["ground_truths"],
            "ground_truths_normalized": [normalize_answer(x) for x in item["ground_truths"]],
            "exact_match": em,
            "f1": f1,
        }
        print(record)
        results.append(record)
        item["image"].close()

predictions = [x["prediction"] for x in results]
ground_truths = [x["ground_truths"] for x in results]
metrics = compute_metrics(predictions, ground_truths)

stats = {
    "dataset_name": DATASET_NAME,
    "dataset_dir": str(DATASET_DIR),
    "test_json": str(TEST_JSON),
    "image_dir": str(IMAGE_DIR),
    "model_variant": MODEL_VARIANT,
    "model_name": MODEL_NAME,
    "device": DEVICE,
    "resolved_device": str(target_device),
    "generation": {
        "temperature": TEMPERATURE,
        "top_p": TOP_P,
        "max_new_tokens": MAX_NEW_TOKENS,
        "do_sample": DO_SAMPLE,
        "batch_size": BATCH_SIZE,
        "min_pixels": MIN_PIXELS,
        "max_pixels": MAX_PIXELS,
    },
    "metrics": metrics,
    "num_predictions": len(results),
    "num_skipped": len(skipped),
    "predictions_path": str(PREDICTIONS_PATH),
}

with open(PREDICTIONS_PATH, "w", encoding="utf-8") as f:
    json.dump({"stats": stats, "predictions": results, "skipped": skipped}, f, ensure_ascii=False, indent=2)

with open(STATS_PATH, "w", encoding="utf-8") as f:
    json.dump(stats, f, ensure_ascii=False, indent=2)

print(json.dumps(stats, ensure_ascii=False, indent=2))
print(f"Saved predictions -> {PREDICTIONS_PATH}")
print(f"Saved stats -> {STATS_PATH}")
results[:3]

Loaded 10,028 test samples from /content/CS2230-VQA/data/ViTextVQA/ViTextVQA_test_gt.json


Running baseline:   0%|          | 0/3 [00:00<?, ?it/s]

A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.


{'question_id': 29, 'image_id': 9, 'image_path': '/content/CS2230-VQA/data/ViTextVQA/st_images/9.jpg', 'question': 'tiệm thuốc này bán kẹo gì ?', 'prompt': '<image>\nUser: Trả lời câu hỏi VQA tiếng Việt dựa trên nội dung trong ảnh.\nHãy trả lời ngắn gọn chỉ bằng một từ, cụm từ hoặc số, không giải thích thêm.\nCâu hỏi: tiệm thuốc này bán kẹo gì ?\nAssistant:', 'prediction': 'tiệm thuốc bán kẹo mát.', 'prediction_normalized': 'tiệm thuốc bán kẹo mát', 'ground_truths': ['cool air'], 'ground_truths_normalized': ['cool air'], 'exact_match': 0.0, 'f1': 0.0}
{'question_id': 30, 'image_id': 9, 'image_path': '/content/CS2230-VQA/data/ViTextVQA/st_images/9.jpg', 'question': 'tiệm thuốc này bán kẹo gì ?', 'prompt': '<image>\nUser: Trả lời câu hỏi VQA tiếng Việt dựa trên nội dung trong ảnh.\nHãy trả lời ngắn gọn chỉ bằng một từ, cụm từ hoặc số, không giải thích thêm.\nCâu hỏi: tiệm thuốc này bán kẹo gì ?\nAssistant:', 'prediction': 'tiệm thuốc bán kẹo mát.', 'prediction_normalized': 'tiệm thuốc bá

A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.


{'question_id': 37, 'image_id': 11, 'image_path': '/content/CS2230-VQA/data/ViTextVQA/st_images/11.jpg', 'question': 'phía xa sau gian hàng dầu ăn bán gì ?', 'prompt': '<image>\nUser: Trả lời câu hỏi VQA tiếng Việt dựa trên nội dung trong ảnh.\nHãy trả lời ngắn gọn chỉ bằng một từ, cụm từ hoặc số, không giải thích thêm.\nCâu hỏi: phía xa sau gian hàng dầu ăn bán gì ?\nAssistant:', 'prediction': 'Select', 'prediction_normalized': 'select', 'ground_truths': ['bánh biscuits'], 'ground_truths_normalized': ['bánh biscuits'], 'exact_match': 0.0, 'f1': 0.0}
{'question_id': 38, 'image_id': 11, 'image_path': '/content/CS2230-VQA/data/ViTextVQA/st_images/11.jpg', 'question': 'mức giá khuyến mãi đầu của bảng đầu tiên là bao nhiêu ?', 'prompt': '<image>\nUser: Trả lời câu hỏi VQA tiếng Việt dựa trên nội dung trong ảnh.\nHãy trả lời ngắn gọn chỉ bằng một từ, cụm từ hoặc số, không giải thích thêm.\nCâu hỏi: mức giá khuyến mãi đầu của bảng đầu tiên là bao nhiêu ?\nAssistant:', 'prediction': '28', 'pr

[{'question_id': 29,
  'image_id': 9,
  'image_path': '/content/CS2230-VQA/data/ViTextVQA/st_images/9.jpg',
  'question': 'tiệm thuốc này bán kẹo gì ?',
  'prompt': '<image>\nUser: Trả lời câu hỏi VQA tiếng Việt dựa trên nội dung trong ảnh.\nHãy trả lời ngắn gọn chỉ bằng một từ, cụm từ hoặc số, không giải thích thêm.\nCâu hỏi: tiệm thuốc này bán kẹo gì ?\nAssistant:',
  'prediction': 'tiệm thuốc bán kẹo mát.',
  'prediction_normalized': 'tiệm thuốc bán kẹo mát',
  'ground_truths': ['cool air'],
  'ground_truths_normalized': ['cool air'],
  'exact_match': 0.0,
  'f1': 0.0},
 {'question_id': 30,
  'image_id': 9,
  'image_path': '/content/CS2230-VQA/data/ViTextVQA/st_images/9.jpg',
  'question': 'tiệm thuốc này bán kẹo gì ?',
  'prompt': '<image>\nUser: Trả lời câu hỏi VQA tiếng Việt dựa trên nội dung trong ảnh.\nHãy trả lời ngắn gọn chỉ bằng một từ, cụm từ hoặc số, không giải thích thêm.\nCâu hỏi: tiệm thuốc này bán kẹo gì ?\nAssistant:',
  'prediction': 'tiệm thuốc bán kẹo mát.',
  'pre

In [ ]:
stats

NameError: name 'stats' is not defined